# PILOT TEST

Pilot Test on 5 000 reviews to measure RoBERTa inference time on Apple Silicon (MPS)

In [1]:
# --- 1. Configuration ---
import time
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PROJECT_ROOT = Path.home() / "Desktop" / "Dissertation_Analysis"
DATA_DIR = PROJECT_ROOT / "data"

MODEL_NAME = "Hello-SimpleAI/chatgpt-detector-roberta"
N_PILOT = 5000
BATCH_SIZE = 32
MAX_LEN = 256

/Users/brg_elise/Desktop/Dissertation_Analysis/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 2. Device detection (MPS for Apple Silicon M-series) ---
if torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"
elif torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = "NVIDIA GPU (CUDA)"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"Device: {device_name}")
print(f"PyTorch version: {torch.__version__}")

Device: Apple Silicon GPU (MPS)
PyTorch version: 2.12.0


In [3]:
# --- 3. Load Model ---
print(f"\nLoading model: {MODEL_NAME}")
print("(First run downloads ~500 MB from Hugging Face)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded.")

# Verify class mapping: index 1 must be the AI class, since we use [:, 1] as P(AI-generated) in the inference loop below
print("Label mapping:", model.config.id2label)
assert model.config.id2label[1] == "ChatGPT", "Model class mapping is not as expected : index 1 is NOT the 'ChatGPT' class - check the model !"


Loading model: Hello-SimpleAI/chatgpt-detector-roberta
(First run downloads ~500 MB from Hugging Face)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 60343.22it/s]


Model loaded.
Label mapping: {0: 'Human', 1: 'ChatGPT'}


In [4]:
# --- 4. Load 5 000 reviews sample ---
print(f"\nLoading {N_PILOT} reviews for pilot test")
df = pd.read_parquet(DATA_DIR / "reviews_clean.parquet")
df_pilot = df.sample(n=N_PILOT, random_state=42).reset_index(drop=True)
texts = df_pilot["text"].tolist()
print(f"Pilot sample: {len(texts)} reviews")

# Check truncation of long reviews (> 256-token)
sample = df["text"].dropna().sample(10000, random_state=42).tolist()
tok_lens = [len(tokenizer.encode(t, truncation=False)) for t in sample]
pct_trunc = np.mean(np.array(tok_lens) > 256)*100
print(f"~{pct_trunc:.2f}% of reviews exceed 256 tokens and are truncated")


Loading 5000 reviews for pilot test


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (599 > 512). Running this sequence through the model will result in indexing errors


Pilot sample: 5000 reviews
~1.14% of reviews exceed 256 tokens and are truncated


Only around **1.14 %** of reviews exceed 256 tokens and are truncated. This has **no impact** on the ongoing analysis

In [5]:
# --- 5. Time Inference ---
print(f"\nRunning inference (batch_size={BATCH_SIZE}, max_len={MAX_LEN})")
start = time.time()
scores = []

with torch.no_grad():
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        ).to(device)
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[:, 1]  # P(AI-generated) -> verified via model.config.id2label
        scores.extend(probs.cpu().numpy().tolist())

        # Log progress every 50 batches (around 1600 reviews)
        if (i // BATCH_SIZE) % 50 == 0 and i > 0:
            elapsed = time.time() - start
            speed = i / elapsed
            print(f"  {i}/{len(texts)} reviews | {speed:.1f} reviews/s")

elapsed = time.time() - start
speed = N_PILOT / elapsed


Running inference (batch_size=32, max_len=256)
  1600/5000 reviews | 69.1 reviews/s
  3200/5000 reviews | 71.3 reviews/s
  4800/5000 reviews | 72.3 reviews/s


In [6]:
# --- 6. Projection for 701 528 reviews ---
TOTAL_REVIEWS = 701_528
projected_seconds = TOTAL_REVIEWS / speed
projected_hours = projected_seconds / 3600

print("\n" + "=" * 60)
print("PILOT RESULTS")
print("=" * 60)
print(f"  Time elapsed:       {elapsed:.1f} seconds")
print(f"  Reviews processed:  {N_PILOT:,}")
print(f"  Speed:              {speed:.1f} reviews/second")
print(f"\n  PROJECTION FOR 701,528 REVIEWS:")
print(f"  Estimated time:     {projected_hours:.2f} hours "
      f"(~{projected_seconds/60:.0f} minutes)")
print("=" * 60)

if projected_hours < 8:
    print("\n  Recommendation: Proceed with full inference (run overnight).")
elif projected_hours < 15:
    print("\n  Recommendation: Borderline. Consider running full inference")
    print("    over a weekend, OR fall back to stratified 200K sample.")
else:
    print("\n  Recommendation: Too long. Fall back to 200K stratified sample.")
    print("    Edit 03_run_roberta.py and set SAMPLE_SIZE = 200_000.")


PILOT RESULTS
  Time elapsed:       68.4 seconds
  Reviews processed:  5,000
  Speed:              73.1 reviews/second

  PROJECTION FOR 701,528 REVIEWS:
  Estimated time:     2.67 hours (~160 minutes)

  Recommendation: Proceed with full inference (run overnight).


This pilot performance test (on a sample of 5 000 reviews) estimated a total processing time of approximately **3 hours**.

This technical validation justified the methodological decision to compute th AI-generated score for the **full corpus** rather tha relying on a reducing sample.

In [7]:
# --- 7. Score distribution preview ---
df_pilot["AI_score_pilot"] = scores
print("\nPilot AI_score distribution:")
print(df_pilot["AI_score_pilot"].describe())

# Sauvegarde pour inspection
df_pilot.to_parquet(DATA_DIR / "pilot_results.parquet", index=False)
print(f"\nPilot data saved to data/pilot_results.parquet for inspection.")


Pilot AI_score distribution:
count    5000.000000
mean        0.111718
std         0.273147
min         0.000244
25%         0.000288
50%         0.000791
75%         0.032072
max         0.999237
Name: AI_score_pilot, dtype: float64

Pilot data saved to data/pilot_results.parquet for inspection.


Result show RoBERTa successfully processed 5 000 pilot reviews.

Around 75 % of observations have an AI score < 3.2%, indicating human origin with high confidence.

In [8]:
## Count "AI-suspect" reviews (score > 0.5)

threshold = 0.5
n_ai_pilot = (df_pilot["AI_score_pilot"] > threshold).sum()
pct = n_ai_pilot / len(df_pilot) * 100
print(f"Reviews AI-suspect (score > {threshold}): {n_ai_pilot}/5000 ({pct:.2f}%)")

Reviews AI-suspect (score > 0.5): 479/5000 (9.58%)


On a sample of 5 000 reviews,only **9.58 %** are considered suspect.

In [9]:
# Visual Analysis of extreme cases
print("Top 3 reviews judged most likely to be AI-generated:")
top = df_pilot.nlargest(3, "AI_score_pilot")[["AI_score_pilot", "text"]]
for _, row in top.iterrows():
    print(f"\nScore: {row['AI_score_pilot']:.4f}")
    print(f"Text:  {row['text'][:300]}...")

print("\n\nTop 3 reviews most likely to have been written by humans:")
bottom = df_pilot.nsmallest(3, "AI_score_pilot")[["AI_score_pilot", "text"]]
for _, row in bottom.iterrows():
    print(f"\nScore: {row['AI_score_pilot']:.4f}")
    print(f"Text:  {row['text'][:300]}...")

Top 3 reviews judged most likely to be AI-generated:

Score: 0.9992
Text:  It looks beautiful and mostly made of plastic and rubbery material. It is quite hard to put three strands of hair into the little clips on top because your hair will not come together as a strand by themselves. You have to watch the videos to see how it works, and really take time to practice to get...

Score: 0.9992
Text:  A little big, the key doesn't fit just right, but it doesn't seem to make a difference. It's a great idea for people with grip problems....

Score: 0.9992
Text:  The gloves are great for the price! 4 stars for thickness, if they were slightly thicker they would hold the heat in more effectively....


Top 3 reviews most likely to have been written by humans:

Score: 0.0002
Text:  [[VIDEOID:e709006f4b529bf3a16633a7b300ea79]] First , I apologize , I didn’t think it would turn out this well and I forgot to take before pictures. This was my first time bleaching and plucking a wig and the hair turn

Linguistic observations:
- Complete, well-structured sentences.
- No spelling errors or missing punctuation.
- Measured tone with varied vocabulary.
- Argumentative structure: observation -> nuance -> recommendation.